fig_plaq — Fig. `fig:plaq` (lossy plaquette dynamics)
Reads `data/` (default) or a fresh `output/` (`USE_OUTPUT = True`) and writes `fig_plaq.pdf`.

In [1]:
USE_OUTPUT = False
OUTPUT_DIR = "output"

In [2]:
import sys
from pathlib import Path
import numpy as np
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
from common.style import plt, W1, style, panel, tag, legend_handles, LABEL, save
if USE_OUTPUT:
    import postprocess
    D = postprocess.build(OUTPUT_DIR)
else:
    D = IO.read_data(HERE / "data")
C = D["curves.csv"]
cur = lambda m, ic, g: IO.select(C, method=m, ic=ic, gamma=g)
ORDER = ["HF", "2B", "TPPX", "mGW", "DSL"]
Z = {"exact": 2, "HF": 3, "2B": 3, "TPPX": 3, "mGW": 3, "DSL": 4}

In [3]:
fig, axg = plt.subplots(2, 2, figsize=(W1, 4.4))
fig.subplots_adjust(left=0.155, right=0.965, top=0.855, bottom=0.095, hspace=0.42, wspace=0.68)
ax = axg.T   # ax[col, row]: columns N, D; letters column-major


def draw(a, ic, g):
    for m in ["exact"] + ORDER:
        r = cur(m, ic, g); a.semilogx(r["t"][1:], r["N"][1:], **style(m), zorder=Z[m])
    a.set_xlim(0.1, 200); a.set_xlabel(r"$tJ$"); a.set_ylabel(r"$N(t)$"); a.set_xticks([1e-1, 1e0, 1e1, 1e2])


a = ax[0, 0]; draw(a, "mixed", 3.0); a.set_ylim(0, 3.4)
tag(a, r"mixed, $\gamma/J=3$", "ll")
r = cur("mGW", "mixed", 3.0); tt, NN = r["t"][1:], r["N"][1:]      # "mGW -> 4" along the mGW curve
k = np.argmin(NN); i0 = k + np.argmax(NN[k:] > 2.55); i1 = k + np.argmax(NN[k:] > 3.3)
fig.canvas.draw()
(x0, y0), (x1, y1) = a.transData.transform([(tt[i0], NN[i0]), (tt[i1], NN[i1])])
ang = np.degrees(np.arctan2(y1 - y0, x1 - x0))
a.text(tt[i0] * 1.25, NN[i0], r"m$GW\to 4$", rotation=ang, rotation_mode="anchor", ha="left", va="top",
       fontsize=8, color=style("mGW")["color"])
ins = a.inset_axes([0.50, 0.60, 0.39, 0.35])                     # <S^2> inset (mGW omitted)
for m in ["exact", "HF", "2B", "TPPX", "DSL"]:
    r = cur(m, "mixed", 3.0); ins.semilogx(r["t"][1:], r["S2"][1:], **style(m, lw=1.0), zorder=Z[m])
ins.set_xlim(0.1, 200); ins.set_xticks([1e-1, 1e0, 1e1, 1e2]); ins.set_xticklabels([]); ins.minorticks_off()
ins.set_yticks([0, 1]); ins.yaxis.tick_right(); ins.spines["left"].set_visible(True); ins.spines["right"].set_visible(True)
ins.tick_params(labelsize=6.5, pad=1.5, length=2)
ins.set_title(r"$\langle\mathbf{S}^2\rangle$", fontsize=7.5, pad=2)
panel(a, "(a)")
a = ax[0, 1]; draw(a, "slater", 10.0); a.set_yscale("log"); a.set_ylim(3e-3, 6)
tag(a, r"Slater, $\gamma/J=10$", "ll"); panel(a, "(b)")


def drawD(a, ic, g, label, loc):
    for m in ["exact"] + ORDER:
        r = cur(m, ic, g); a.plot(r["t"][1:], r["D"][1:], **style(m), zorder=Z[m])
    a.set_xscale("log"); a.set_yscale("symlog", linthresh=1e-3, linscale=0.6)
    a.axhline(0, color="#52514e", lw=0.6, zorder=1)
    a.set_xlim(0.1, 100); a.set_xlabel(r"$tJ$"); a.set_ylabel(r"$D(t)$")
    a.set_yticks([-1, -1e-2, 0, 1e-2, 1]); a.set_ylim(-3, 3)
    a.set_yticklabels([r"$-1$", r"$-10^{-2}$", "0", r"$10^{-2}$", "1"])
    a.set_xticks([1e-1, 1e0, 1e1, 1e2])
    tag(a, label, loc)


drawD(ax[1, 0], "one_pair", 1.0, r"one pair, $\gamma/J=1$", "ll"); panel(ax[1, 0], "(c)")
drawD(ax[1, 1], "cdw", 3.0, r"CDW, $\gamma/J=3$", "lr"); panel(ax[1, 1], "(d)")
M = ["exact"] + ORDER
fig.legend(legend_handles(M), [LABEL[m] for m in M], loc="upper center", ncol=3, bbox_to_anchor=(0.55, 1.0),
           columnspacing=1.2)
fig.align_ylabels(axg[:, 0]); fig.align_ylabels(axg[:, 1])
save(fig, HERE / "fig_plaq")